# Adresli NER modeli: BERTurk fine-tuning

Bu çalışma, WikiNER kişi/kurum/yer cümlelerini Türkiye’den gerçek OpenStreetMap adres kayıtlarıyla birleştirir. Model `PERSON`, `ORG`, `LOCATION` ve `ADDRESS` etiketleriyle eğitilir. Adres kayıtları tek başına adres satırlarıdır; doğal mesaj içindeki adres performansını ölçmez.

## Başlamadan önce
1. Colab’da T4 GPU çalışma zamanını seç.
2. İlk hücrede bilgisayarından `real-address-v2-colab.zip` dosyasını yükle.
3. Aşağıdaki hücreleri sırayla çalıştır. Eğitim yaklaşık ilk eğitimle benzer süre alabilir.

In [ ]:
from google.colab import files

uploaded = files.upload()  # real-address-v2-colab.zip dosyasını seç
archive_name = 'real-address-v2-colab.zip'
if archive_name not in uploaded:
    raise FileNotFoundError(f'{archive_name} yüklenmedi')
import zipfile

with zipfile.ZipFile(archive_name) as zf:
    zf.extractall('/content')
print('Proje /content/bitirme_maskeleme içine açıldı')

In [ ]:
%cd /content/bitirme_maskeleme
!nvidia-smi
!pip -q install -r training/ner/requirements.txt

In [ ]:
!python training/ner/train.py --data-dir data/ner/processed/real-address-v2 --labels-path training/ner/labels.json --validate-only

In [ ]:
!python training/ner/train.py \
  --data-dir data/ner/processed/real-address-v2 \
  --labels-path training/ner/labels.json \
  --output-dir /content/ner-with-address-model \
  --epochs 3 \
  --seed 42

## Sonuç ve indirme

Çıktıda `test_openstreetmap_real_addresses` bölümü adres satırlarına ait test sonuçlarını, `test_wiki` bölümü kişi/kurum/yer performansını ayrı gösterir. Bu ayrım, adresleri tanımayı öğrenirken önceki NER kabiliyetinin korunup korunmadığını görmemizi sağlar. Her iki ölçüm de kendi kaynak dağılımı içinde yorumlanmalı; OSM testi doğal mesaj başarısı değildir.

Model dosyalarını indirmek için aşağıdaki hücreyi çalıştır. ZIP yaklaşık 450 MB olabilir.

In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

out = Path('/content/ner-with-address-model')
archive = Path('/content/ner-with-address-model.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_STORED) as z:
    for name in ('metrics.json', 'manifest.json'):
        z.write(out/name, name)
    for p in (out/'model').rglob('*'):
        if p.is_file(): z.write(p, p.relative_to(out))
print('Arşiv boyutu (MB):', round(archive.stat().st_size / 1024**2, 1))
files.download(str(archive))

## Cümle içindeki adresleri ayrıca ölç

Aşağıdaki ek kontrol, aynı kaynakta farklı holdout split’e ayrılmış gerçek OSM adreslerini sabit cümle kalıplarına yerleştirir. Cümleler yapaydır; bu sonuç doğal mesaj başarısı değildir. Adresin çevresindeki metin eklendiğinde modelin sınırları bulup bulmadığını gösteren ek bir kontroldür.

In [ ]:
from google.colab import files

uploaded = files.upload()  # address-context-eval-colab.zip
eval_zip = next(name for name in uploaded if name.startswith('address-context-eval-colab'))
with zipfile.ZipFile(eval_zip) as zf: zf.extractall('/content')
!python training/ner/evaluate_model.py --model-dir /content/ner-with-address-model/model --data-dir data/ner/processed/openstreetmap-tr/sentence-context-v1 --split test --output-json /content/ner-with-address-model/context-test-metrics.json